# Physics-Informed Neural Network (PINN) for Fluid-Structure Interaction
## 1. Environment Setup

In [1]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import numpy as np
import time

gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print(f"GPU Details: {gpu}")
else:
    print("NO GPU DETECTED. TensorFlow is running on the CPU.")

os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
tf.config.optimizer.set_jit(False) # Disable the Just-In-Time compiler

# Import custom modules
from models.cfd_pinn import CFDPINN
from utils.data_loader import parse_tfrecord, tf_process_wrapper
from utils.physics_losses import compute_physics_residual

## 2. Load and Preprocess Dataset

In [2]:
file_pattern = "./AirfoilMNIST-incompressible-train.tfrecord-*"
filepaths = tf.io.gfile.glob(file_pattern)

raw_dataset = tf.data.TFRecordDataset(filepaths, num_parallel_reads=tf.data.AUTOTUNE)
raw_dataset = raw_dataset.ignore_errors()
print(f"Total TFRecord files located: {len(filepaths)}")

parsed_dataset = raw_dataset.map(parse_tfrecord)

## 3. Define Hyperparameters and Build Data Stream

In [3]:
EPOCHS = 1000
BATCH_SIZE = 2048  
LEARNING_RATE = 1e-3

W_DATA = tf.constant(1.0, dtype=tf.float32)     
W_PHYS = tf.constant(0.001, dtype=tf.float32)   
NU = tf.constant(0.01, dtype=tf.float32)

print("Building Training Stream...")
train_dataset = parsed_dataset.map(tf_process_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.unbatch()
train_dataset = train_dataset.shuffle(10000)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
print("Training Stream Ready!")

## 4. Initialize CFD Model

In [4]:
cfd_model = CFDPINN(num_hidden_layers=4, neurons_per_layer=32)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
mse_loss_fn = tf.keras.losses.MeanSquaredError()

dummy_input = tf.constant([[0.5, 0.1, 0.2, 0.3]], dtype=tf.float32)
dummy_output = cfd_model(dummy_input)

print(f"Input Shape (X, Y): {dummy_input.shape}")
print(f"Output Shape (u, v, p): {dummy_output.shape}")

# Load pre-trained weights if available
try:
    cfd_model.load_weights('./cfd_checkpoints/stage1gb_1e-4_epoch_31.weights.h5')
    print("Weights loaded successfully.")
except:
    print("No pre-trained weights found.")

## 5. Define Training Step

In [5]:
@tf.function
def train_step(coords_full, u_true, v_true, p_true):
    x_batch = coords_full[:, 0:1]
    y_batch = coords_full[:, 1:2]
    angle_batch = coords_full[:, 2:3]
    mach_batch = coords_full[:, 3:4]
    
    with tf.GradientTape() as tape_weights:
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch([x_batch, y_batch])
            with tf.GradientTape(persistent=True) as tape1:
                tape1.watch([x_batch, y_batch])
                
                inputs = tf.concat([x_batch, y_batch, angle_batch, mach_batch], axis=1)
                preds = cfd_model(inputs)
                
                u_pred = preds[:, 0:1]
                v_pred = preds[:, 1:2]
                p_pred = preds[:, 2:3]
                
            u_x = tape1.gradient(u_pred, x_batch)
            u_y = tape1.gradient(u_pred, y_batch)
            v_x = tape1.gradient(v_pred, x_batch)
            v_y = tape1.gradient(v_pred, y_batch)
            p_x = tape1.gradient(p_pred, x_batch)
            p_y = tape1.gradient(p_pred, y_batch)
            del tape1 
            
        u_xx = tape2.gradient(u_x, x_batch)
        u_yy = tape2.gradient(u_y, y_batch)
        v_xx = tape2.gradient(v_x, x_batch)
        v_yy = tape2.gradient(v_y, y_batch)
        del tape2 
        
        eq_continuity = u_x + v_y
        eq_mom_x = (u_pred * u_x + v_pred * u_y) + p_x - NU * (u_xx + u_yy)
        eq_mom_y = (u_pred * v_x + v_pred * v_y) + p_y - NU * (v_xx + v_yy)
        
        loss_data = mse_loss_fn(u_true, u_pred) + mse_loss_fn(v_true, v_pred) + mse_loss_fn(p_true, p_pred)
        
        loss_phys = tf.reduce_mean(tf.square(eq_continuity)) + \
                    tf.reduce_mean(tf.square(eq_mom_x)) + \
                    tf.reduce_mean(tf.square(eq_mom_y))
                    
        total_loss = (W_DATA * loss_data) + (W_PHYS * loss_phys)
        
    gradients = tape_weights.gradient(total_loss, cfd_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, cfd_model.trainable_variables))
    
    return total_loss, loss_data, loss_phys

## 6. Training Execution Loop

In [6]:
print("Initiating CFD Surrogate Training Loop...")

checkpoint_dir = './cfd_checkpoints'
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

for epoch in range(EPOCHS):
    start_time = time.time()
    
    epoch_loss_total = 0.0
    epoch_loss_data = 0.0
    epoch_loss_phys = 0.0
    batches = 0
    
    for batch_coords, batch_labels in train_dataset:
        
        u_t = batch_labels[:, 0:1]
        v_t = batch_labels[:, 1:2]
        p_t = batch_labels[:, 2:3]
        
        t_loss, d_loss, p_loss = train_step(batch_coords, u_t, v_t, p_t)
        
        epoch_loss_total += t_loss
        epoch_loss_data += d_loss
        epoch_loss_phys += p_loss
        batches += 1
        
    avg_total = epoch_loss_total / batches
    avg_data = epoch_loss_data / batches
    avg_phys = epoch_loss_phys / batches
    
    epoch_duration = time.time() - start_time
    
    if epoch % 1 == 0:
        print(f"Epoch {epoch:04d} | {epoch_duration:.2f}s | Total Loss: {avg_total:.6f} | Data Loss: {avg_data:.6f} | Phys Loss: {avg_phys:.6f}")

    if epoch % 50 == 0:
        filepath = os.path.join(checkpoint_dir, f'cfd_model_epoch_{epoch}.weights.h5')
        cfd_model.save_weights(filepath)

print("Training Complete.")

### 7. FEA Initialization & Geometry Extraction

In [ ]:
from models.fea_pinn import FEAPINN

print("Setting up FSI extraction framework...")

def extract_solid_domain(mask_np):
    """Extracts internal and skin points for the solid FEA solver based on the mask."""
    x_range = np.linspace(-1, 1, 256)
    y_range = np.linspace(-1, 1, 256)
    X_grid, Y_grid = np.meshgrid(x_range, y_range, indexing='xy')
    
    X_flat, Y_flat = X_grid.flatten(), Y_grid.flatten()
    mask_flat = mask_np.flatten()
    
    # Solid domain is where mask == 0.0
    is_solid = (mask_flat == 0.0)
    X_solid = X_flat[is_solid].reshape(-1, 1)
    Y_solid = Y_flat[is_solid].reshape(-1, 1)
    
    # Create tensor
    pts_solid_tf = tf.convert_to_tensor(np.hstack((X_solid, Y_solid)), dtype=tf.float32)
    return X_grid, Y_grid, pts_solid_tf, is_solid

# Test Dataset Iterator for FSI Visualization
test_pattern = "./AirfoilMNIST-incompressible-validation.tfrecord-*"
test_filepaths = tf.io.gfile.glob(test_pattern)
raw_test_dataset = tf.data.TFRecordDataset(test_filepaths).map(parse_tfrecord)
fsi_viz_iter = iter(raw_test_dataset)

### 8. FEA Physics Engine (Hooke's Law)

In [ ]:

# Material Properties (Aluminum-like)
E = tf.constant(69e9, dtype=tf.float32)
NU_SOLID = tf.constant(0.3, dtype=tf.float32)
LAMBDA = (E * NU_SOLID) / ((1.0 + NU_SOLID) * (1.0 - 2.0 * NU_SOLID))
MU = E / (2.0 * (1.0 + NU_SOLID))

@tf.function
def train_fea_step(model, opt, pts, pressure_load):
    x = pts[:, 0:1]
    y = pts[:, 1:2]
    
    with tf.GradientTape() as tape_weights:
        with tf.GradientTape(persistent=True) as tape_spatial:
            tape_spatial.watch([x, y])
            
            inputs = tf.concat([x, y], axis=1)
            preds = model(inputs)
            
            dx, dy = preds[:, 0:1], preds[:, 1:2]
            s_xx, s_yy, t_xy = preds[:, 2:3], preds[:, 3:4], preds[:, 4:5]
            
        # Kinematics (Strains)
        dx_x = tape_spatial.gradient(dx, x)
        dy_y = tape_spatial.gradient(dy, y)
        dx_y = tape_spatial.gradient(dx, y)
        dy_x = tape_spatial.gradient(dy, x)
        
        # Stress Gradients (Equilibrium)
        s_xx_x = tape_spatial.gradient(s_xx, x)
        t_xy_y = tape_spatial.gradient(t_xy, y)
        t_xy_x = tape_spatial.gradient(t_xy, x)
        s_yy_y = tape_spatial.gradient(s_yy, y)
        del tape_spatial
        
        # Constitutive Law (Hooke's Law) Residuals
        eq_sxx = s_xx - ((LAMBDA + 2.0 * MU) * dx_x + LAMBDA * dy_y)
        eq_syy = s_yy - ((LAMBDA + 2.0 * MU) * dy_y + LAMBDA * dx_x)
        eq_txy = t_xy - (MU * (dx_y + dy_x))
        
        # Equilibrium Residuals (Static)
        eq_fx = s_xx_x + t_xy_y
        eq_fy = t_xy_x + s_yy_y
        
        # Interior Loss (Heavily weighted to prevent "Volume Locking" / Squeezing)
        loss_int = tf.reduce_mean(tf.square(eq_sxx)) + tf.reduce_mean(tf.square(eq_syy)) + \
                   tf.reduce_mean(tf.square(eq_txy)) + tf.reduce_mean(tf.square(eq_fx)) + \
                   tf.reduce_mean(tf.square(eq_fy))
        
        # Boundary Condition Loss (Simplified mapping to pressure load)
        # Assuming pressure acts entirely on the solid domain boundaries for this test
        loss_bc = tf.reduce_mean(tf.square(s_yy + pressure_load)) 
        
        total_loss = (100.0 * loss_bc) + (10.0 * loss_int) # 10x weight to enforce stiffness
        
    grads = tape_weights.gradient(total_loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    
    return total_loss

### 9. Solve FSI for a specific Airfoil Scenario

In [ ]:
print("Fetching geometry and solving CFD load...")

# 1. Grab test geometry
mask, p_gt, u_gt, v_gt, angle, mach, label_raw = next(fsi_viz_iter)
mask_np = mask.numpy().T
config_name = label_raw.numpy().decode('utf-8')

# 2. Extract solid domain
X_grid, Y_grid, pts_solid_tf, is_solid = extract_solid_domain(mask_np)

# 3. Ask CFD PINN for the pressure field
X_flat = X_grid.flatten().reshape(-1, 1)
Y_flat = Y_grid.flatten().reshape(-1, 1)
angle_v = tf.fill([X_flat.shape[0], 1], angle)
mach_v = tf.fill([X_flat.shape[0], 1], mach)

cfd_input = tf.concat([X_flat, Y_flat, angle_v, mach_v], axis=1)
cfd_preds = cfd_model(cfd_input, training=False)
p_pred_grid = cfd_preds[:, 2].numpy().reshape(256, 256)

# Map predicted pressure to the solid points
p_solid_load = tf.convert_to_tensor(p_pred_grid.flatten()[is_solid].reshape(-1, 1), dtype=tf.float32)

# 4. Initialize and Train FEA Solver for this specific load
print(f"Solving Solid Mechanics for {config_name}...")
tf.keras.backend.clear_session()
fea_model = FEAPINN()
opt_fea = tf.keras.optimizers.Adam(learning_rate=1e-3)

# Warmup to prevent tf.function variable creation error
_ = fea_model(tf.zeros((1, 2)))
opt_fea.build(fea_model.trainable_variables)

# Quick solver loop (1000 epochs takes ~15-20s on GPU)
epochs_fea = 1000
for epoch in range(epochs_fea):
    loss = train_fea_step(fea_model, opt_fea, pts_solid_tf, p_solid_load)
    if epoch % 250 == 0:
        print(f"FEA Epoch {epoch} | Loss: {loss.numpy():.4e}")

print("FEA Solution Converged!")

### 10. 2x2 FSI Comparative Visualization

In [ ]:

print("Generating complete FSI visual report...")

# Get structural predictions
preds_fea = fea_model(pts_solid_tf, training=False)
dx, dy = preds_fea[:, 0].numpy(), preds_fea[:, 1].numpy()
s_xx, s_yy, t_xy = preds_fea[:, 2].numpy(), preds_fea[:, 3].numpy(), preds_fea[:, 4].numpy()

# Derived physical quantities
disp_mag = np.sqrt(dx**2 + dy**2)
von_mises = np.sqrt(s_xx**2 - s_xx*s_yy + s_yy**2 + 3*(t_xy**2))

# Deformation scaling for visual clarity (Scaling vertical bending heavily)
DEFORMATION_SCALE = 100.0 
X_def = pts_solid_tf[:, 0].numpy() + (dx * 1.0) 
Y_def = pts_solid_tf[:, 1].numpy() + (dy * DEFORMATION_SCALE)

# Plotting
fig, axs = plt.subplots(2, 2, figsize=(18, 16))
phys_cmap = 'jet' 

# 1. Top-Left: Normal Mask (Geometry)
axs[0, 0].imshow(mask_np, extent=[-1, 1, -1, 1], origin='lower', cmap='gray')
axs[0, 0].set_title("Input Geometry (Airfoil Mask)")
axs[0, 0].set_xlim([-0.7, 0.7]); axs[0, 0].set_ylim([-0.5, 0.5])
axs[0, 0].set_aspect('equal')

# 2. Top-Right: Predicted Pressure Magnitude
# Mask the fluid plot for clear visuals
p_plot_masked = np.copy(p_pred_grid)
p_plot_masked[mask_np == 0] = np.nan
im2 = axs[0, 1].imshow(p_plot_masked, extent=[-1, 1, -1, 1], origin='lower', cmap='RdBu_r')
axs[0, 1].contour(mask_np, levels=[0.5], colors='black', extent=[-1, 1, -1, 1], linewidths=1.5)
axs[0, 1].set_title("Predicted CFD Pressure Load")
axs[0, 1].set_xlim([-1, 1]); axs[0, 1].set_ylim([-1, 1])
axs[0, 1].set_aspect('equal')
plt.colorbar(im2, ax=axs[0, 1], label="Pressure", fraction=0.046, pad=0.04)

# 3. Bottom-Left: Von Mises Stress with Geometry Overlay
axs[1, 0].imshow(mask_np, extent=[-1, 1, -1, 1], origin='lower', cmap='gray', alpha=0.3)
sc3 = axs[1, 0].scatter(X_def, Y_def, c=von_mises, cmap=phys_cmap, s=8)
axs[1, 0].set_title(f"Von Mises Stress (Deformation {DEFORMATION_SCALE}x)")
axs[1, 0].set_xlim([-0.7, 0.7]); axs[1, 0].set_ylim([-0.5, 0.5])
axs[1, 0].set_aspect('equal'); axs[1, 0].grid(True, linestyle='--', alpha=0.3)
plt.colorbar(sc3, ax=axs[1, 0], label="Stress", fraction=0.046, pad=0.04)

# 4. Bottom-Right: Displacement Plot with Geometry Overlay
axs[1, 1].imshow(mask_np, extent=[-1, 1, -1, 1], origin='lower', cmap='gray', alpha=0.3)
sc4 = axs[1, 1].scatter(X_def, Y_def, c=disp_mag, cmap=phys_cmap, s=8)
axs[1, 1].set_title(f"Displacement Magnitude (Deformation {DEFORMATION_SCALE}x)")
axs[1, 1].set_xlim([-0.7, 0.7]); axs[1, 1].set_ylim([-0.5, 0.5])
axs[1, 1].set_aspect('equal'); axs[1, 1].grid(True, linestyle='--', alpha=0.3)
plt.colorbar(sc4, ax=axs[1, 1], label="Displacement", fraction=0.046, pad=0.04)

fig.suptitle(f"FSI Visual Report: {config_name}", fontsize=18, y=0.97)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
plt.show()